### Landmark data checks

Verifies the 28 sparse landmark configurations before any downstream analysis, and
records the symmetry numbers reported in the methods. Nothing here is written back
to the landmark files -- the other notebooks load the same `.mrk.json` files
directly and analyse them in full.

* superimposition: centroid size, consensus vs atlas mean, distance to consensus
* which coordinate index is the axis of bilateral symmetry, and which are y and z
* symmetric / asymmetric partition: how much directional and fluctuating asymmetry
* whether symmetrising would change anything downstream (PC scores, disparity ranks)

*Last edited by K. Wolcott on 19 Sep 2026*

### Setup and paths

In [ ]:
# Imports and paths
import os, re, json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from NSM.plotting import load_mrk_json
from NSM.morphometrics import (centroid_size, dist_to_mean, gm_prcomp, morphol_disparity,
                               mshape, procrustes_dist, two_d_array, lm_diff, check_axis_labels)

# Specify training directory and atlas directory
RUN          = "run_v72"                         # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"            # atlas/builder run that produced alignedLMs
DROPBOX_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
SPECIES_CSV  = "../lizard_species_list.csv"

# Build other directories relative to those above
cwd       = Path.cwd()
base_wd   = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

LM_DIR      = DROPBOX_ROOT / ATLAS_RUN / "alignedLMs"
ATLAS_DIR   = DROPBOX_ROOT / ATLAS_RUN / "atlas"
MEAN_LMS_FN = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"

OUT_DIR = Path("lm_check_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

# Load config and mesh filenames
config_path = "model_params_config.json"
with open(config_path) as f:
    cfg = json.load(f)
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"\033[92mLoaded config from {config_path}\033[0m -- {len(all_vtk_files)} meshes")

In [ ]:
# Family labels -- only needed for the disparity rank check at the end
pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed]})

sdf = pd.read_csv(SPECIES_CSV)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

### Load landmarks (`.mrk.json`) and build the shape array

In [ ]:
# Load 3D Slicer Atlas aligned and scaled landmark data
lm_coords = []
for fpath in all_vtk_files:
    lm_name = os.path.splitext(fpath)[0] + ".mrk.json"
    lm_path = LM_DIR / lm_name
    coords, _ = load_mrk_json(lm_path)
    lm_coords.append(coords)

lm_coords_3d = np.stack(lm_coords)   # (N, p, 3)
print(f"Landmark data shape - 3d: {lm_coords_3d.shape}")

# Atlas mean sparse landmarks (same landmark set as LM_DIR, from the setup cell)
mean_lms_3d, _ = load_mrk_json(MEAN_LMS_FN)
print(f"Atlas mean landmarks shape: {mean_lms_3d.shape}")

### Superimposition and sizing checks

In [ ]:
# Data checks before analysis
CV_THRESHOLD = 5.0        # % — above this the configurations aren't on a common scale

cs = np.array([centroid_size(X) for X in lm_coords_3d])
cv = 100 * cs.std() / cs.mean()
print(f"Centroid size: mean={cs.mean():.5f}  sd={cs.std():.5f}  CV={cv:.2f}%  "
      f"range=({cs.min():.5f}, {cs.max():.5f})")

consensus = mshape(lm_coords_3d)
print(f"\nConsensus vs atlas mean landmarks: Procrustes distance = "
      f"{procrustes_dist(consensus, mean_lms_3d):.6f}")
print(f"Mean per-landmark offset = {np.linalg.norm(consensus - mean_lms_3d, axis=1).mean():.6f}")

d_mean = dist_to_mean(lm_coords_3d)
print(f"\nProcrustes distance to consensus: mean={d_mean.mean():.5f}  "
      f"median={np.median(d_mean):.5f}  max={d_mean.max():.5f}")

# Scale only — a non-unit CV means the input was not fully Procrustes-aligned,
# so rotation is suspect too and GPA should be re-run before the symmetry partition
if cv > CV_THRESHOLD:
    print(f"\n\033[33mCV {cv:.2f}% exceeds {CV_THRESHOLD}% — rescaling to unit centroid size. "
          f"Re-run GPA before trusting anything downstream.\033[0m")
    lm_coords_3d = np.stack([(X - X.mean(axis=0)) / centroid_size(X) for X in lm_coords_3d])
    consensus = mshape(lm_coords_3d)
    print(f"Rescaled: CV now {100 * np.std([centroid_size(X) for X in lm_coords_3d]):.4f}%")

### Identify coordinate axis order/rotation (how do XYZ correspond to anatomic planes)

In [ ]:
# Identify which coordinate index is x (the axis of bilateral symmetry), y, and z

midline_idx = [20, 21, 8, 9, 12, 15]                                   # TO DO: midline landmarks (0-based)
paired_landmarks = {0: 5, 1: 4, 2: 3, 18: 19, 22: 23, 24: 25,
                    6: 7, 26: 27, 13: 14, 10: 11, 16: 17}              # TO DO: left: right pairs (0-based)
COTYLE_IDX, NEURAL_SPINE_IDX = 15, 12                                  # TO DO: for the y-axis check
ZYG_PR_IDX, ZYG_PO_IDX = 16, 0                                         # TO DO: for the z-axis check

# 1st check - axis of symmetry: midline landmarks are near-constant along x
x_idx = check_axis_labels("1st Check", "x", np.var(lm_coords_3d[:, midline_idx, :].reshape(-1, 3), axis=0), "min", 
                          f"Midline Landmarks {midline_idx}")

# 2nd check - the same axis should carry most of the left-right variation
lr_diffs = np.concatenate([lm_diff(lm_coords_3d, l, r) for l, r in paired_landmarks.items()], axis=0)
x_idx2 = check_axis_labels("2nd Check", "x", np.var(lr_diffs, axis=0), "max",
                  f"Paired left-right landmarks (mirrored across X) {paired_landmarks}")

# 3rd check - cotyle vs neural spine separate along y
y_idx = check_axis_labels("3rd Check", "y", np.mean(np.abs(lm_diff(lm_coords_3d, NEURAL_SPINE_IDX, COTYLE_IDX)), axis=0), "max",
                          f"Cotyle vs neural spine landmarks ({COTYLE_IDX} & {NEURAL_SPINE_IDX})")

# 4th check - pre- vs post-zygapophyses separate along z
z_idx = check_axis_labels("4th Check", "z", np.mean(np.abs(lm_diff(lm_coords_3d, ZYG_PO_IDX, ZYG_PR_IDX)), axis=0), "max",
                          f"Pre- and post-zygapophyses landmarks ({ZYG_PR_IDX} & {ZYG_PO_IDX})")

assert x_idx == x_idx2, f"symmetry axis disagrees between checks 1 and 2: {x_idx} vs {x_idx2}"
assert len({x_idx, y_idx, z_idx}) == 3, f"axis checks disagree: x={x_idx} y={y_idx} z={z_idx}"

# Build axis list using determined order
axis_order = [None, None, None]
axis_order[z_idx], axis_order[x_idx], axis_order[y_idx] = 'z', 'x', 'y'
print("\nAxis order:", axis_order)

### Symmetric/Asymmetric components of variance (similar to R geomorph bilat.symmetry)

In [ ]:
# Symmetric vs asymmetric, after re-superimposing each mirror on its original.

# Ordinary Procrustes Analysis: simple pairwise alignment, where bilat.symmetry in R
# geomorph runs GPA over originals and mirrors jointly. Without it, the small shared
# rotation left by GPA to a common consensus is counted as directional asymmetry.
def opa(X, Y):
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    U, _, Vt = np.linalg.svd(Yc.T @ Xc)
    R = U @ np.diag([1, 1, np.sign(np.linalg.det(U @ Vt))]) @ Vt
    return Yc @ R + X.mean(0)

# Mirror across the axis of symmetry and relabel the pairs, then re-superimpose
reflected = lm_coords_3d.copy()
reflected[:, :, x_idx] *= -1
for l, r in paired_landmarks.items():
    reflected[:, [l, r], :] = reflected[:, [r, l], :]
mirrored = np.stack([opa(X, M) for X, M in zip(lm_coords_3d, reflected)])

symm_component = (lm_coords_3d + mirrored) / 2.0
asym_component = (lm_coords_3d - mirrored) / 2.0
da_component   = asym_component.mean(axis=0, keepdims=True)     # directional asymmetry
fa_component   = asym_component - da_component                  # fluctuating asymmetry

# Sums of squares about each component's own mean, so all three are on one scale
n      = len(lm_coords_3d)
sym_ss = ((symm_component - symm_component.mean(axis=0)) ** 2).sum()
da_ss  = n * (da_component ** 2).sum()
fa_ss  = (fa_component ** 2).sum()
da_ratio, fa_ratio = da_ss / sym_ss, fa_ss / sym_ss

print(f"DA / among-specimen symmetric shape variance: {100*da_ratio:.2f}%")
print(f"FA / among-specimen symmetric shape variance: {100*fa_ratio:.2f}%")
print("NB: per-specimen OPA removes rigid asymmetry, so DA is near zero by construction;\n"
      "    FA is the informative quantity here.")

# Diagnostic: how much of the uncorrected DA was a rigid tilt of the symmetry plane
da_raw   = ((lm_coords_3d - reflected) / 2.0).mean(axis=0)
C        = symm_component.mean(0)
A        = np.column_stack([C[:, y_idx], C[:, z_idx], np.ones(len(C))])
coef, *_ = np.linalg.lstsq(A, da_raw[:, x_idx], rcond=None)
resid    = da_raw[:, x_idx] - A @ coef
da_ss_raw = n * (da_raw ** 2).sum()
r2 = 1 - (resid ** 2).sum() / ((da_raw[:, x_idx] - da_raw[:, x_idx].mean()) ** 2).sum()
print(f"\nWithout OPA: DA = {100*da_ss_raw/sym_ss:.0f}% of symmetric variance, "
      f"R^2 = {r2:.3f} against a rigid tilt (coefs y, z = {coef[:2].round(4)})")

### Would using the symmetric component/symmetrizing coordinates change downstream results compring LM's with NSM?

In [ ]:
# Principal components: raw vs symmetric landmark configurations
pca_raw, pca_sym = gm_prcomp(lm_coords_3d), gm_prcomp(symm_component)
pc_rho = []
for i in range(4):
    r = abs(spearmanr(pca_raw["x"][:, i], pca_sym["x"][:, i]).statistic)
    pc_rho.append(r)
    print(f"PC{i+1}: |rho|={r:.4f}  {100*pca_raw['prop'][i]:.2f}% -> {100*pca_sym['prop'][i]:.2f}%")

In [ ]:
# Family disparity ranks: raw vs symmetric landmark configurations
MIN_N = 5      # matches morphol_disparity_lms_latents.ipynb

keep   = specimens["family"].notna().values
counts = specimens.loc[keep, "family"].value_counts()
keep  &= ~specimens["family"].isin(counts[counts < MIN_N].index).values
groups = specimens.loc[keep, "family"].values

ranks = {}
for name, Y in (("symmetric", two_d_array(symm_component)), ("full", two_d_array(lm_coords_3d))):
    v, _, _ = morphol_disparity(Y[keep], groups, iter=999)
    ranks[name] = v.rank(ascending=False).astype(int)

sym, ful = ranks["symmetric"], ranks["full"].reindex(ranks["symmetric"].index)
rank_rho, rank_p = spearmanr(sym, ful)
moved = (sym - ful)[lambda s: s != 0]
print(f"rho = {rank_rho:.4f}, p = {rank_p:.2e}")
print(f"{len(moved)} of {len(sym)} families moved, max |shift| = {moved.abs().max()}")
print(f"ranks of the families that moved (check they are not at the extremes): "
      f"{sorted(moved.index.map(sym.to_dict()))}")

### Numbers reported in Mats & methods

In [ ]:
summary = pd.Series({
    "n_specimens":            len(lm_coords_3d),
    "n_landmarks":            lm_coords_3d.shape[1],
    "centroid_size_cv_pct":   cv,
    "consensus_vs_atlas_d":   procrustes_dist(consensus, mean_lms_3d),
    "mean_d_to_consensus":    d_mean.mean(),
    "median_d_to_consensus":  np.median(d_mean),
    "max_d_to_consensus":     d_mean.max(),
    "DA_pct_of_sym":          100 * da_ratio,
    "FA_pct_of_sym":          100 * fa_ratio,
    "DA_pct_no_opa":          100 * da_ss_raw / sym_ss,
    "tilt_r2_no_opa":         r2,
    "min_PC_rho_raw_vs_sym":  min(pc_rho),
    "family_rank_rho":        rank_rho,
    "n_families_moved":       len(moved),
    "max_family_rank_shift":  int(moved.abs().max()) if len(moved) else 0,
}, name="value")

summary.to_csv(OUT_DIR / "landmark_checks_summary.csv")
print(summary.to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nSaved -> {(OUT_DIR / 'landmark_checks_summary.csv').resolve()}")